# 17. Recursion

After this notebook you'll be able to:

- write a function that calls itself,
- find the two parts every such function needs: the **base case** and the **smaller step**,
- follow the calls going down and the answers coming back up,
- fix `RecursionError` when it happens.

This is one of the harder ideas in programming, so this notebook goes slowly. Run every cell, and read the tables. Most people need to see it a few times before it clicks. That's normal.

## Russian dolls

You've seen Russian dolls: open a doll, and there's a smaller doll inside. Open that one, and there's a smaller one again. You keep going until the tiny doll in the middle, which doesn't open. Then you're done.

Every step is the same job ("open this doll") on a **smaller** doll. And there's a **last** doll that stops it all.

A function that calls itself works the same way. This is called **recursion**. Each call does a small part of the work and hands a smaller version of the problem to another call of the same function. One small case is answered straight away, and that stops it.

---

## First: a function waits for the one it calls

Before a function calls **itself**, remember what happens when a function calls **another** function. Run this:

In [ ]:
def make_tea():
    print("    make_tea: boil water")
    print("    make_tea: add tea")

def morning():
    print("morning starts")
    make_tea()
    print("morning carries on: drink tea")

morning()

```
morning starts
    make_tea: boil water
    make_tea: add tea
morning carries on: drink tea
```

When `morning` reaches `make_tea()`, it **pauses**. `make_tea` runs from start to end. Only then does `morning` carry on from the line after the call.

Hold on to that rule: **the caller waits until the call finishes, then carries on from where it stopped.** Everything in this notebook follows from it.

---

## A function that calls itself

In [ ]:
def countdown(n):
    if n == 0:
        print("Go!")
        return
    print(n)
    countdown(n - 1)

countdown(3)

```
3
2
1
Go!
```

Here's every call, one row each:

| call | `n` | `n == 0`? | what it does |
| --- | --- | --- | --- |
| `countdown(3)` | 3 | no | prints `3`, then calls `countdown(2)` |
| `countdown(2)` | 2 | no | prints `2`, then calls `countdown(1)` |
| `countdown(1)` | 1 | no | prints `1`, then calls `countdown(0)` |
| `countdown(0)` | 0 | **yes** | prints `Go!` and returns. No more calls. |

Each call is a smaller doll: `n` goes 3, 2, 1, 0. `countdown(0)` is the tiny doll that doesn't open.

**Try it:** call `countdown(5)` instead.

### Predict, then run

This one is almost the same, but the `print` comes **after** the call. What will `count_up(3)` print? Write your guess down before running it.

In [ ]:
def count_up(n):
    if n == 0:
        return
    count_up(n - 1)
    print(n)

count_up(3)

```
1
2
3
```

Did you guess `3 2 1`? Most people do. Here's why it's `1 2 3`, using the waiting rule:

| what happens | the calls waiting |
| --- | --- |
| `count_up(3)` starts. Before printing, it calls `count_up(2)` and **waits**. | 3 |
| `count_up(2)` starts, calls `count_up(1)`, waits. | 3, 2 |
| `count_up(1)` starts, calls `count_up(0)`, waits. | 3, 2, 1 |
| `count_up(0)`: `n == 0`, returns straight away. | 3, 2, 1 |
| `count_up(1)` carries on: prints **1**, finishes. | 3, 2 |
| `count_up(2)` carries on: prints **2**, finishes. | 3 |
| `count_up(3)` carries on: prints **3**, finishes. | (none) |

Nothing is printed on the way **down**. The prints happen on the way back **up**, and the last call to start is the first to finish.

## The two parts every recursive function needs

1. **The base case**: the small case you answer straight away, without calling yourself. In `countdown`, it's `n == 0`. It's the tiny doll.
2. **The smaller step**: call yourself on a **smaller** problem. In `countdown`, it's `countdown(n - 1)`. Every call must get **closer** to the base case.

Write the base case first, at the top of the function. If either part is missing, the calls never stop. You'll see what that looks like further down.

---

## Returning answers

Most of the time the function **returns** something, and each call builds its answer from the smaller call's answer.

### Sum of a list

The sum of `[3, 1, 4]` is "the first item, plus the sum of the rest". The rest, `[1, 4]`, is a smaller list. The sum of an empty list is 0: that's the base case.

In [ ]:
def total(nums):
    if len(nums) == 0:
        return 0
    return nums[0] + total(nums[1:])

print(total([3, 1, 4]))

```
8
```

`nums[1:]` is the list without its first item, so every call gets a shorter list until it's empty.

**Going down.** Each call can't finish its sum yet, so it waits:

| call | `nums[0]` | waits for |
| --- | --- | --- |
| `total([3, 1, 4])` | 3 | `total([1, 4])` |
| `total([1, 4])` | 1 | `total([4])` |
| `total([4])` | 4 | `total([])` |
| `total([])` | | nothing: base case, returns **0** |

**Coming back up.** Now each waiting call gets its answer and finishes, from the bottom:

| call | works out | returns |
| --- | --- | --- |
| `total([])` | | 0 |
| `total([4])` | 4 + 0 | 4 |
| `total([1, 4])` | 1 + 4 | 5 |
| `total([3, 1, 4])` | 3 + 5 | **8** |

### Watching it happen

Here's the same function with prints added, so you can watch. `depth` counts how deep we are, only so that each level is pushed further right: `"    " * depth` is 4 spaces repeated `depth` times.

In [ ]:
def total(nums, depth):
    pad = "    " * depth
    print(pad + f"total({nums}) starts")
    if len(nums) == 0:
        print(pad + "empty list: return 0")
        return 0
    answer = nums[0] + total(nums[1:], depth + 1)
    print(pad + f"total({nums}) returns {answer}")
    return answer

print(total([3, 1, 4], 0))

```
total([3, 1, 4]) starts
    total([1, 4]) starts
        total([4]) starts
            total([]) starts
            empty list: return 0
        total([4]) returns 4
    total([1, 4]) returns 5
total([3, 1, 4]) returns 8
8
```

The lines going right are the calls going **down**. The lines coming back left are the answers coming **up**. It's the two tables above, printed by Python itself.

**Try it:** change the list to `[5, 10, 20, 1]` and follow the output.

### Factorial

The **factorial** of 4, written 4!, is `4 × 3 × 2 × 1 = 24`. Notice that `4!` is `4 × 3!`, and `3!` is `3 × 2!`. Each one is "this number times the factorial of the one below". The base case: `1!` is 1.

In [ ]:
def factorial(n):
    if n == 1:
        return 1
    return n * factorial(n - 1)

print(factorial(4))
print(factorial(5))

```
24
120
```

And with prints, the same way as `total`:

In [ ]:
def factorial_traced(n, depth):
    pad = "    " * depth
    print(pad + f"factorial({n}) starts")
    if n == 1:
        print(pad + "base case: return 1")
        return 1
    answer = n * factorial_traced(n - 1, depth + 1)
    print(pad + f"factorial({n}) returns {answer}")
    return answer

print(factorial_traced(4, 0))

```
factorial(4) starts
    factorial(3) starts
        factorial(2) starts
            factorial(1) starts
            base case: return 1
        factorial(2) returns 2
    factorial(3) returns 6
factorial(4) returns 24
24
```

The same thing as a picture:

```
  going down (each call waits)            coming back up (answers return)

  factorial(4) = 4 * factorial(3)  ....   4 * 6  = 24   <- final answer
     factorial(3) = 3 * factorial(2) ...  3 * 2  = 6
        factorial(2) = 2 * factorial(1)   2 * 1  = 2
           factorial(1) = 1  ─────────────────┘
           (base case: stop going down)
```

**Try it:** run `factorial_traced(6, 0)` and check the last answer is 720.

### The call stack

Remember the stack of plates from the stacks notebook? Python keeps the waiting calls in a pile just like that, called the **call stack**.

- Each new call goes on **top** of the pile.
- When a call returns, it's taken off the top, and the call underneath carries on from where it was waiting.
- The last call to start is the first to finish.

```
  factorial(1)   <- on top: returns 1, comes off first
  factorial(2)   <- waiting for factorial(1)
  factorial(3)   <- waiting for factorial(2)
  factorial(4)   <- at the bottom: started first, finishes last
```

---

## How to write one yourself: three questions

When you write a recursive function, ask these three, in this order:

1. **What's the smallest input, and its answer?** That's the base case.
2. **If a call on a slightly smaller input already gave me the right answer, how do I finish the job?** That's the smaller step. Trust the smaller call; don't try to follow it all the way down in your head.
3. **Does every call really get smaller, so it reaches the base case?**

Let's use them on "the length of a string", without `len`:

1. Smallest input: the empty string `""`. Its length is 0.
2. If `length(s[1:])` (the string without its first letter) is right, the full length is `1 + length(s[1:])`.
3. `s[1:]` is one letter shorter each time, so it reaches `""`.

In [ ]:
def length(s):
    if s == "":
        return 0
    return 1 + length(s[1:])

print(length("tea"))
print(length(""))

```
3
0
```

| call | works out | returns |
| --- | --- | --- |
| `length("tea")` | 1 + `length("ea")` | 1 + 2 = **3** |
| `length("ea")` | 1 + `length("a")` | 1 + 1 = 2 |
| `length("a")` | 1 + `length("")` | 1 + 0 = 1 |
| `length("")` | base case | 0 |

---

## Recursion inside `class Solution`

On LeetCode your function is a **method** inside `class Solution`. To call the method from inside itself, write **`self.`** in front of its name:

In [ ]:
class Solution:
    def fact(self, n: int) -> int:
        if n == 1:
            return 1
        return n * self.fact(n - 1)

print(Solution().fact(4))

```
24
```

Leave out `self.` and Python can't find it. Run this:

In [ ]:
class Solution:
    def fact(self, n: int) -> int:
        if n == 1:
            return 1
        return n * fact(n - 1)

print(Solution().fact(4))

```
NameError: name 'fact' is not defined
```

A method lives inside the class, so on its own the name `fact` means nothing. Write `self.fact(n - 1)`.

---

## Common mistakes

### No base case: `RecursionError`

Leave out the base case, and the calls never stop going down. Run this. The error is long; scroll to the last line.

In [ ]:
def factorial_broken(n):
    return n * factorial_broken(n - 1)

print(factorial_broken(4))

The last line says:

```
RecursionError: maximum recursion depth exceeded
```

`factorial_broken(4)` called 3, then 2, 1, 0, -1, -2, ... with nothing to stop it. Python allows about 1000 waiting calls on the pile. After that it gives up with **`RecursionError`**: "too many calls inside calls".

When you see it, check two things: **is there a base case**, and **does every call really get closer to it?** Writing `countdown(n)` instead of `countdown(n - 1)` has the same problem: `n` never changes, so it never reaches 0.

### Forgetting `return` in front of the call

Run this:

In [ ]:
def factorial_no_return(n):
    if n == 1:
        return 1
    n * factorial_no_return(n - 1)

print(factorial_no_return(3))

```
TypeError: unsupported operand type(s) for *: 'int' and 'NoneType'
```

The last line works out `n * ...` and then throws the answer away. So `factorial_no_return(2)` returns nothing, which is `None`. Then the call above it tries `3 * None`, and that's the error. It must be `return n * factorial_no_return(n - 1)`.

---

# ✏️ Your turn

Every task here must use recursion: the function calls itself. For each one, answer the three questions first (smallest input? how to finish from the smaller answer? does it get smaller?), then write the code.

### Task 1: add up 1 to n

Write `sum_to(n)`. It returns `1 + 2 + ... + n`. `sum_to(4)` is `4 + 3 + 2 + 1 = 10`. `sum_to(0)` is 0.

<details><summary>Hint</summary>

`sum_to(4)` is `4 + sum_to(3)`. What's the smallest `n`?
</details>

In [ ]:
def sum_to(n):
    # your code here
    pass

In [ ]:
from checks import check
check(sum_to(4), 10)
check(sum_to(1), 1)
check(sum_to(0), 0)
check(sum_to(10), 55)

### Task 2: power

Write `power(base, exp)`. It returns `base` multiplied by itself `exp` times, without using `**`. `power(2, 5)` is `2 × 2 × 2 × 2 × 2 = 32`. Anything to the power 0 is 1.

<details><summary>Hint</summary>

`power(2, 5)` is `2 * power(2, 4)`. Which `exp` is the smallest?
</details>

In [ ]:
def power(base, exp):
    # your code here
    pass

In [ ]:
check(power(2, 5), 32)
check(power(3, 0), 1)
check(power(5, 1), 5)
check(power(10, 3), 1000)

### Task 3: add up the digits, recursively

Write `digit_sum(n)`. `digit_sum(4072)` is `4 + 0 + 7 + 2 = 13`. This time, no loop: call yourself.

<details><summary>Hint</summary>

`n % 10` is the last digit and `n // 10` is the rest of the number. So `digit_sum(4072)` is `2 + digit_sum(407)`. When is a number small enough to answer straight away?
</details>

In [ ]:
def digit_sum(n):
    # your code here
    pass

In [ ]:
check(digit_sum(4072), 13)
check(digit_sum(7), 7)
check(digit_sum(0), 0)
check(digit_sum(99), 18)

### Task 4: reverse a string

Write `reverse(s)`. It returns the string back to front: `"tea"` gives `"aet"`. No slicing with `[::-1]`: call yourself.

<details><summary>Hint</summary>

The reverse of `"tea"` is the reverse of `"ea"`, with `"t"` stuck on the **end**.
</details>

In [ ]:
def reverse(s):
    # your code here
    pass

In [ ]:
check(reverse("tea"), "aet")
check(reverse("a"), "a")
check(reverse(""), "")
check(reverse("Tanvi"), "ivnaT")

### Task 5: palindrome

A **palindrome** reads the same both ways, like `"level"`. Write `is_palindrome(s)` that returns `True` or `False`.

<details><summary>Hint</summary>

If the first and last letters are different, it's not a palindrome. If they're the same, the answer is whether the middle, `s[1:-1]`, is a palindrome. A string of length 0 or 1 is always a palindrome.
</details>

In [ ]:
def is_palindrome(s):
    # your code here
    pass

In [ ]:
check(is_palindrome("level"), True)
check(is_palindrome("abba"), True)
check(is_palindrome("tea"), False)
check(is_palindrome("a"), True)
check(is_palindrome(""), True)

### Task 6: the biggest number

Write `biggest(nums)` for a list that has **at least one** number. Return the biggest one, without `max`.

<details><summary>Hint</summary>

Base case: a list of one number. Otherwise, find the biggest of the rest, `nums[1:]`, and compare it with `nums[0]`.
</details>

In [ ]:
def biggest(nums):
    # your code here
    pass

In [ ]:
check(biggest([3, 9, 2]), 9)
check(biggest([5]), 5)
check(biggest([-4, -1, -7]), -1)
check(biggest([1, 2, 3, 4]), 4)

### Task 7: LeetCode 231, Power of Two

Return `True` if `n` is a power of two (1, 2, 4, 8, 16, ...), otherwise `False`. `n` can be zero or negative.

<details><summary>Hint</summary>

1 is a power of two. Zero, negatives and odd numbers (other than 1) are not. Otherwise, `n` is a power of two if `n // 2` is. Remember `self.`.
</details>

In [ ]:
class Solution:
    def isPowerOfTwo(self, n: int) -> bool:
        # your code here
        pass

In [ ]:
check(Solution().isPowerOfTwo(1), True)
check(Solution().isPowerOfTwo(16), True)
check(Solution().isPowerOfTwo(3), False)
check(Solution().isPowerOfTwo(6), False)
check(Solution().isPowerOfTwo(0), False)
check(Solution().isPowerOfTwo(-8), False)

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the check cell under it.

**Question 1.** In `countdown`, which part is the base case?

- A) `print(n)`
- B) `if n == 0:` and what it does
- C) `countdown(n - 1)`

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("17-q1", answer_1)

**Question 2.** You see `RecursionError: maximum recursion depth exceeded`. What's the most likely cause?

- A) There's no base case, or the calls don't get smaller
- B) A typo in a `print`

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("17-q2", answer_2)

**Question 3.** In `count_up(3)`, the `print` comes after the recursive call. What is printed **first**?

- A) `3`
- B) `1`

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("17-q3", answer_3)

**Question 4.** In `factorial_no_return`, the last line has no `return`. What does `factorial_no_return(2)` give back?

- A) `2`
- B) `0`
- C) `None`

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("17-q4", answer_4)

---

## Done

You can now write a function that calls itself, with a base case and a smaller step. You can follow the calls going down and the answers coming back up, call a method recursively with `self.`, and fix a `RecursionError`.

Stuck on a task? The answers are in `solutions/17-recursion.ipynb`. Try for real first.

**Next:** recursion in real problems: two calls at once, helper functions inside a LeetCode method, and making it fast with `@cache` (`18-recursion-in-practice`).